# IFCasFormer — CAVE Dataset Inference
End-to-end inference: PSNR · SSIM · SAM · ERGAS

In [42]:
import subprocess, sys
for pkg in ['einops', 'fvcore']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('deps ok')


deps ok


In [43]:
import os, sys, math, glob, types, datetime, warnings, logging, random
from collections import OrderedDict
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from einops import rearrange, repeat
from einops.layers.torch import Rearrange
from torch.nn.init import _calculate_fan_in_and_fan_out
from math import exp

os.environ['CUDA_DEVICE_ORDER']   = 'PCI_BUS_ID'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
torch.backends.cudnn.enabled   = True
torch.backends.cudnn.benchmark = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)


Using device: cuda


In [44]:
# ─── Dataset paths ──────────────────────────────────────────
CAVE_TEST_DIR = "/kaggle/input/cave-dataset-3/cave_test"
MASK_PATH = "/kaggle/input/casformer-mask/mask_test.mat"  # Check if your mask is named this or mask_test.mat
MODEL_PTH = "/kaggle/input/ifcasformer/pytorch/default/1/cave_test.pth"
OUT_DIR = "/kaggle/working/results/"
os.makedirs(OUT_DIR, exist_ok=True)

USE_AUTHORS_FORMAT = True
print("Dataset: authors pre-packaged cave_test/ -- CORRECT FORMAT")


Dataset: authors pre-packaged cave_test/ -- CORRECT FORMAT


## SSIM (ssim_torch.py — verbatim from repo)

In [45]:
import torch
import torch.nn.functional as F
from torch.autograd import Variable
import numpy as np
from math import exp

def gaussian(window_size, sigma):
    gauss = torch.Tensor([exp(-(x - window_size // 2) ** 2 / float(2 * sigma ** 2)) for x in range(window_size)])
    return gauss / gauss.sum()

def create_window(window_size, channel):
    _1D_window = gaussian(window_size, 1.5).unsqueeze(1)
    _2D_window = _1D_window.mm(_1D_window.t()).float().unsqueeze(0).unsqueeze(0)
    window = Variable(_2D_window.expand(channel, 1, window_size, window_size).contiguous())
    return window

def _ssim(img1, img2, window, window_size, channel, size_average=True):
    mu1 = F.conv2d(img1, window, padding=window_size // 2, groups=channel)
    mu2 = F.conv2d(img2, window, padding=window_size // 2, groups=channel)

    mu1_sq = mu1.pow(2)
    mu2_sq = mu2.pow(2)
    mu1_mu2 = mu1 * mu2

    sigma1_sq = F.conv2d(img1 * img1, window, padding=window_size // 2, groups=channel) - mu1_sq
    sigma2_sq = F.conv2d(img2 * img2, window, padding=window_size // 2, groups=channel) - mu2_sq
    sigma12 = F.conv2d(img1 * img2, window, padding=window_size // 2, groups=channel) - mu1_mu2

    C1 = 0.01 ** 2
    C2 = 0.03 ** 2

    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))

    if size_average:
        return ssim_map.mean()
    else:
        return ssim_map.mean(1).mean(1).mean(1)

class SSIM(torch.nn.Module):
    def __init__(self, window_size=11, size_average=True):
        super(SSIM, self).__init__()
        self.window_size = window_size
        self.size_average = size_average
        self.channel = 1
        self.window = create_window(window_size, self.channel)

    def forward(self, img1, img2):
        (_, channel, _, _) = img1.size()

        if channel == self.channel and self.window.data.type() == img1.data.type():
            window = self.window
        else:
            window = create_window(self.window_size, channel)

            if img1.is_cuda:
                window = window.cuda(img1.get_device())
            window = window.type_as(img1)

            self.window = window
            self.channel = channel

        return _ssim(img1, img2, window, self.window_size, channel, self.size_average)

def ssim(img1, img2, window_size=11, size_average=True):
    (_, channel, _, _) = img1.size()
    window = create_window(window_size, channel)

    if img1.is_cuda:
        window = window.cuda(img1.get_device())
    window = window.type_as(img1)

    return _ssim(img1, img2, window, window_size, channel, size_average)


## Metric & utility functions (from utils.py)

In [46]:
# ── torch_psnr (utils.py) ────────────────────────────────────────────
def torch_psnr(img, ref):
    img = (img * 256).round()
    ref = (ref * 256).round()
    nC = img.shape[0]
    psnr = 0
    for i in range(nC):
        mse = torch.mean((img[i, :, :] - ref[i, :, :]) ** 2)
        psnr += 10 * torch.log10((255 * 255) / mse)
    return psnr / nC

# ── torch_ssim (utils.py) ────────────────────────────────────────────
def torch_ssim(img, ref):
    return ssim(torch.unsqueeze(img, 0), torch.unsqueeze(ref, 0))

# ── SAM_GPU (utils.py) ───────────────────────────────────────────────
def SAM_GPU(img, ref):
    C = img.size()[0]
    H = img.size()[1]
    W = img.size()[2]
    esp = 1e-12
    Itrue = img.clone()
    Ifake = ref.clone()
    nom = torch.mul(Itrue, Ifake).sum(dim=0)
    denominator = Itrue.norm(p=2, dim=0, keepdim=True).clamp(min=esp) * \
                  Ifake.norm(p=2, dim=0, keepdim=True).clamp(min=esp)
    denominator = denominator.squeeze()
    sam = torch.div(nom, denominator).acos()
    sam[sam != sam] = 0
    sam_sum = torch.sum(sam) / (H * W) / np.pi * 180
    return sam_sum

# ── ERGAS (standard metric, not in original repo) ────────────────────
def ERGAS(img, ref, scale=4):
    """img, ref: numpy [H,W,C] in [0,1]"""
    C = img.shape[2]
    ergas = 0.0
    for c in range(C):
        rmse = np.sqrt(np.mean((img[:,:,c] - ref[:,:,c])**2))
        mean_ref = np.mean(ref[:,:,c]) + 1e-12
        ergas += (rmse / mean_ref)**2
    return 100 / scale * math.sqrt(ergas / C)


In [47]:
# ── shift / shift_back / mask utilities (utils.py) ──────────────────
def shift(inputs, step=2):
    [bs, nC, row, col] = inputs.shape
    output = torch.zeros(bs, nC, row, col + (nC - 1) * step).to(inputs.device).float()
    for i in range(nC):
        output[:, i, :, step * i:step * i + col] = inputs[:, i, :, :]
    return output

def shift_back_meas(inputs, step=2):  # utils.py version: 3D meas -> 4D HSI
    [bs, row, col] = inputs.shape
    nC = 28
    output = torch.zeros(bs, nC, row, col - (nC - 1) * step).to(inputs.device).float()
    for i in range(nC):
        output[:, i, :, :] = inputs[:, :, step * i:step * i + col - (nC - 1) * step]
    return output

def generate_masks(mask_path, batch_size, data_type='cave'):
    if data_type == 'cave':
        mask = sio.loadmat(mask_path)['CASSI']
    elif data_type == 'kaist':
        mask = sio.loadmat(mask_path)['mask']
    elif data_type == 'icvl':
        mask = sio.loadmat(mask_path)['mask']
    # mask_cave.mat is stored as [H, H+(nC-1)*step] (pre-extended CASSI width).
    # gen_meas_torch needs a square [H, H] mask — shift() re-extends it later.
    H = mask.shape[0]
    mask = mask[:H, :H]
    mask3d = np.tile(mask[:, :, np.newaxis], (1, 1, 28))
    mask3d = np.transpose(mask3d, [2, 0, 1])
    mask3d = torch.from_numpy(mask3d)
    [nC, H, W] = mask3d.shape
    mask3d_batch = mask3d.expand([batch_size, nC, H, W]).cuda().float()
    return mask3d_batch

def init_mask(mask_path, input_mask_type, batch_size, data_type='cave'):
    mask3d_batch = generate_masks(mask_path, batch_size, data_type)
    if input_mask_type == 'Phi':
        shift_mask3d_batch = shift(mask3d_batch)
        input_mask = shift_mask3d_batch
    elif input_mask_type == 'Phi_PhiPhiT':
        input_mask = mask3d_batch  # simplified
    elif input_mask_type == 'Mask':
        input_mask = mask3d_batch
    elif input_mask_type is None:
        input_mask = None
    return mask3d_batch, input_mask

def gen_meas_torch(data_batch, mask3d_batch, Y2H=True, mul_mask=False):
    [batch_size, nC, H, W] = data_batch.shape
    mask3d_batch = (mask3d_batch[0, :, :, :]).expand([batch_size, nC, H, W]).cuda().float()
    temp = shift(mask3d_batch * data_batch)
    meas = torch.sum(temp, 1)
    if Y2H:
        meas = meas / nC * 2
        H_out = shift_back_meas(meas)
        if mul_mask:
            HM = torch.mul(H_out, mask3d_batch)
            return HM
        return H_out
    return meas

def time2file_name(time):
    year = time[0:4]; month = time[5:7]; day = time[8:10]
    hour = time[11:13]; minute = time[14:16]; second = time[17:19]
    return year+'_'+month+'_'+day+'_'+hour+'_'+minute+'_'+second


## Dataset loading (LoadTest from utils.py — adapted for separate HSI/RGB dirs)

In [48]:
def LoadTest(data_type="cave"):
    if not os.path.exists(CAVE_TEST_DIR):
        raise FileNotFoundError(f"Dataset not found at {CAVE_TEST_DIR}")
    scene_list = sorted(os.listdir(CAVE_TEST_DIR))
    test_data = np.zeros((len(scene_list), 512, 512, 28)).astype(np.float32)
    test_rgb  = np.zeros((len(scene_list), 512, 512, 3)).astype(np.float32)
    names = []
    for i, scene in enumerate(scene_list):
        data = sio.loadmat(os.path.join(CAVE_TEST_DIR, scene))
        img  = data["cave_data"].astype(np.float32)   # [512,512,28]
        rgb  = data["cave_rgb"].astype(np.float32)    # [512,512,3]
        test_data[i, :, :, :] = img
        test_rgb [i, :, :, :] = rgb
        names.append(scene)
        print(f"  [{i+1:02d}] {scene}  HSI{img.shape}  RGB{rgb.shape}")
    test_data = torch.from_numpy(np.transpose(test_data, (0, 3, 1, 2)))
    test_rgb  = torch.from_numpy(np.transpose(test_rgb,  (0, 3, 1, 2)))
    return test_data, test_rgb, names

print("Loading CAVE test set ...")
test_data, label_rgb, scene_names = LoadTest(data_type="cave")
print(f"HSI tensor : {test_data.shape}")
print(f"RGB tensor : {label_rgb.shape}")


Loading CAVE test set ...
  [01] scene01.mat  HSI(512, 512, 28)  RGB(512, 512, 3)
  [02] scene04.mat  HSI(512, 512, 28)  RGB(512, 512, 3)
  [03] scene05.mat  HSI(512, 512, 28)  RGB(512, 512, 3)
  [04] scene06.mat  HSI(512, 512, 28)  RGB(512, 512, 3)
  [05] scene07.mat  HSI(512, 512, 28)  RGB(512, 512, 3)
  [06] scene08.mat  HSI(512, 512, 28)  RGB(512, 512, 3)
  [07] scene09.mat  HSI(512, 512, 28)  RGB(512, 512, 3)
  [08] scene10.mat  HSI(512, 512, 28)  RGB(512, 512, 3)
  [09] scene21.mat  HSI(512, 512, 28)  RGB(512, 512, 3)
  [10] scene27.mat  HSI(512, 512, 28)  RGB(512, 512, 3)
HSI tensor : torch.Size([10, 28, 512, 512])
RGB tensor : torch.Size([10, 3, 512, 512])


## CASSI Mask setup

In [49]:
# ── CASSI Mask setup ────────────────────────────────────────────────────────
# MASK_PATH is already defined in the Paths cell [03].
# Do NOT redefine it here — just validate and load.
print("Using mask:", MASK_PATH)

if not os.path.exists(MASK_PATH):
    raise FileNotFoundError(
        f"mask_cave.mat not found at {MASK_PATH}\n"
        "Download it from the README Google Drive link:\n"
        "  https://drive.google.com/drive/folders/1vQaPOj3oYZCq6s09useXcYvofhI6YLOD\n"
        "Upload as a Kaggle dataset and update MASK_PATH in the Paths cell."
    )

N_SCENES = test_data.shape[0]
mask3d_batch, input_mask = init_mask(
    MASK_PATH, input_mask_type='Phi', batch_size=N_SCENES, data_type='cave')
print("mask3d_batch:", mask3d_batch.shape)
print("input_mask  :", input_mask.shape)


Using mask: /kaggle/input/casformer-mask/mask_test.mat
mask3d_batch: torch.Size([10, 28, 512, 512])
input_mask  : torch.Size([10, 28, 512, 566])


## Network model — HRFT.py (verbatim from repo)

In [50]:
import torch.nn as nn
import torch
import torch.nn.functional as F
from einops import rearrange, repeat
from einops.layers.torch import Rearrange
import math
import warnings
from torch.nn.init import _calculate_fan_in_and_fan_out
_dataset_type = "cave"  # renamed: avoids shadowing test_data tensor

def _no_grad_trunc_normal_(tensor, mean, std, a, b):
    def norm_cdf(x):
        return (1. + math.erf(x / math.sqrt(2.))) / 2.

    if (mean < a - 2 * std) or (mean > b + 2 * std):
        warnings.warn("mean is more than 2 std from [a, b] in nn.init.trunc_normal_. "
                      "The distribution of values may be incorrect.",
                      stacklevel=2)
    with torch.no_grad():
        l = norm_cdf((a - mean) / std)
        u = norm_cdf((b - mean) / std)
        tensor.uniform_(2 * l - 1, 2 * u - 1)
        tensor.erfinv_()
        tensor.mul_(std * math.sqrt(2.))
        tensor.add_(mean)
        tensor.clamp_(min=a, max=b)
        return tensor

def pair(t):
    return t if isinstance(t, tuple) else (t, t)

class PreNorm(nn.Module):
    def __init__(self, dim, fn):
        super().__init__()
        self.fn = fn
        self.norm = nn.LayerNorm(dim)

    def forward(self, x, *args, **kwargs):
        x = self.norm(x)
        return self.fn(x, *args, **kwargs)

class GELU(nn.Module):
    def forward(self, x):
        return F.gelu(x)

def conv(in_channels, out_channels, kernel_size, bias=False, padding=1, stride=1):
    return nn.Conv2d(
        in_channels, out_channels, kernel_size,
        padding=(kernel_size // 2), bias=bias, stride=stride)

def shift_back(inputs, step=2):
    [bs, nC, row, col] = inputs.shape
    if row == col:
        return inputs
    else:
        if _dataset_type == "cave":
            down_sample = 512 // row
        elif _dataset_type=="kaist":
            down_sample = 256 // row
        elif _dataset_type=="icvl":
            down_sample = 1300 // row
        step = float(step) / float(down_sample * down_sample)
        out_col = row
        for i in range(nC):
            inputs[:, i, :, :out_col] = \
                inputs[:, i, :, int(step * i):int(step * i) + out_col]
        return inputs[:, :, :, :out_col]

# ----------------------------------------
#       Mask-Guided Attention
# ----------------------------------------
class MaskGuidedMechanism(nn.Module):
    def __init__(
            self, n_feat):
        super(MaskGuidedMechanism, self).__init__()
        self.conv1 = nn.Conv2d(n_feat, n_feat, kernel_size=1, bias=True)
        self.conv2 = nn.Conv2d(n_feat, n_feat, kernel_size=1, bias=True)
        self.depth_conv = nn.Conv2d(n_feat, n_feat, kernel_size=5, padding=2, bias=True, groups=n_feat)

    def forward(self, mask_shift):
        # x: b,c,h,w
        [bs, nC, row, col] = mask_shift.shape
        mask_shift = self.conv1(mask_shift)
        attn_map = torch.sigmoid(self.depth_conv(self.conv2(mask_shift)))
        res = mask_shift * attn_map
        mask_shift = res + mask_shift
        mask_emb = shift_back(mask_shift)
        mask_emb = mask_emb.permute(0, 2, 3, 1)
        return mask_emb

# ----------------------------------------
#       Mspe Transformer Block
# ----------------------------------------
class Mspe(nn.Module):
    def __init__(
            self,
            dim, heads,
            dim_head
    ):
        super().__init__()
        self.num_heads = heads
        self.dim_head = dim_head
        self.to_q1 = nn.Linear(dim, dim_head * heads, bias=False)
        self.to_k1 = nn.Linear(dim, dim_head * heads, bias=False)
        self.to_v1 = nn.Linear(dim, dim_head * heads, bias=False)
        self.rescale = nn.Parameter(torch.ones(heads, 1, 1)).cuda()
        self.proj = nn.Linear(dim_head * heads, dim, bias=True)
        self.band_emb = nn.Sequential(
            nn.Conv2d(dim, dim, 3, 1, 1, bias=False, groups=dim),
            GELU(),
            nn.Conv2d(dim, dim, 3, 1, 1, bias=False, groups=dim),
        )
        self.MaskGuidedMechanism = MaskGuidedMechanism(dim)

    def forward(self, x, mask):
        """
        x_in: [b,h,w,c]
        mask: [1,c,h,w]
        return out: [b,h,w,c]
        """
        b, h, w, c = x.shape
        x = x.reshape(b, h * w, c)
        q1_inp = self.to_q1(x)
        k1_inp = self.to_k1(x)
        v1_inp = self.to_v1(x)
        mask_attn = self.MaskGuidedMechanism(mask)
        if b != 0:
            mask_attn = (mask_attn[0, :, :, :]).expand([b, h, w, c])
        q1, k1, v1, mask_attn = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h=self.num_heads),
                                    (q1_inp, k1_inp, v1_inp, mask_attn.flatten(1, 2)))
        v1 = v1 * mask_attn
        q1 = q1.transpose(-2, -1)
        k1 = k1.transpose(-2, -1)
        v1 = v1.transpose(-2, -1)
        q1 = F.normalize(q1, dim=-1, p=2)
        k1 = F.normalize(k1, dim=-1, p=2)
        attn = (k1 @ q1.transpose(-2, -1))
        attn = attn * self.rescale.cuda()
        attn = attn.softmax(dim=-1)
        x = attn @ v1
        x = x.permute(0, 3, 1, 2)
        x = x.reshape(b, h * w, self.num_heads * self.dim_head)
        out_c = self.proj(x).view(b, h, w, c)
        out_p = self.band_emb(v1_inp.reshape(b, h, w, c).permute(0, 3, 1, 2)).permute(0, 2, 3, 1)
        Xspe = out_c + out_p

        return Xspe

class SpeFE(nn.Module):
    def __init__(self, dim):
        super(SpeFE, self).__init__()
        self.dim = dim
        self.conv_11 = nn.Conv2d(in_channels=dim, out_channels=dim, kernel_size=3, padding=1)
        self.LeakyReLU = nn.LeakyReLU(dim)
        self.conv_12 = nn.Conv2d(in_channels=dim, out_channels=dim, kernel_size=3, padding=1)

    def forward(self, LR_HSI_Up):
        ln_11 = nn.LayerNorm(LR_HSI_Up.shape).cuda()
        ln_12 = nn.LayerNorm(LR_HSI_Up.shape).cuda()
        out1_1 = self.LeakyReLU(ln_11(self.conv_11(LR_HSI_Up)))
        out1_2 = self.LeakyReLU(ln_12(self.conv_12(out1_1)))
        LR_HSI = out1_2 + LR_HSI_Up

        return LR_HSI_Up


# --------------------------------------------------------------------------------
#           Spatial Feature Extractor (SpaFE)——————RGB+RGB(DW,UP)
# --------------------------------------------------------------------------------
class SpaFE(nn.Module):
    def __init__(self, n_fts=28):
        super(SpaFE, self).__init__()
        self.n_fts = n_fts
        lv1_c = int(n_fts)
        lv2_c = int(n_fts * 2)
        lv4_c = int(n_fts * 4)
        # 3 256 256 ->28 256 256
        self.layer_1 = nn.Sequential(nn.Conv2d(in_channels=3, out_channels=lv1_c, kernel_size=3, padding=1),
                                     nn.BatchNorm2d(lv1_c),
                                     nn.LeakyReLU(negative_slope=0.0),
                                     )
        # 3 256 256 -> 56 128 128
        self.layer_2 = nn.Sequential(nn.Conv2d(in_channels=3, out_channels=lv2_c, kernel_size=3, padding=1),
                                     nn.BatchNorm2d(lv2_c),
                                     nn.LeakyReLU(negative_slope=0.0),
                                     nn.MaxPool2d(kernel_size=2, stride=2),
                                     )
        # 3 256 256 -> 112 64 64
        self.layer_3 = nn.Sequential(nn.Conv2d(in_channels=3, out_channels=lv2_c, kernel_size=3, padding=1),
                                     nn.BatchNorm2d(lv2_c),
                                     nn.LeakyReLU(negative_slope=0.0),
                                     nn.MaxPool2d(kernel_size=2, stride=2),
                                     nn.Conv2d(in_channels=lv2_c, out_channels=lv4_c, kernel_size=3, padding=1),
                                     nn.BatchNorm2d(lv4_c),
                                     nn.LeakyReLU(negative_slope=0.0),
                                     nn.MaxPool2d(kernel_size=2, stride=2)
                                     )
    def forward(self, x_rgb):
        x1 = self.layer_1(x_rgb)
        x2 = self.layer_2(x_rgb)
        x3 = self.layer_3(x_rgb)

        return [x1, x2, x3]

# --------------------------------------------------------------------------------
#          Spatial-Spetral Cross-Attention
# -------------------------------------------------------------------------------
class MulCorssAttention(nn.Module):
    def __init__(self, dim, heads, dim_head=64, token_height=16, token_width=16, q_bias=False, k_bias=False,
                 v_bias=False, proj_drop=0.):
        super().__init__()
        self.heads = 8
        self.scale = dim_head ** -0.5
        self.token_height = token_height
        self.token_width = token_width
        self.dim = dim
        self.to_q2 = nn.Linear(token_height * token_width * 2, token_height * token_width * 2, q_bias)
        self.to_k2 = nn.Linear(token_height * token_width * 2, token_height * token_width * 2, k_bias)
        self.to_v2 = nn.Linear(token_height * token_width * 2, token_height * token_width * 2, v_bias)
        self.to_out = nn.Sequential(
            nn.Linear(token_height * token_width * 2, token_height * token_width * dim),
            nn.Dropout(proj_drop),
        )
        self.norm = nn.LayerNorm(dim)
        self.attend = nn.Softmax(dim=-1)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, V_2_in, K_2_in, Q_2_in):
        B, C, H, W = Q_2_in.shape
        (image_height, image_width) = (H, W)
        assert image_height % self.token_height == 0 and image_width % self.token_width == 0, 'Image dimensions must be divisible by the patch size.'
        patch_num_x = image_height // self.token_height
        patch_num_y = image_width // self.token_width
        num_patches = (image_height // self.token_height) * (image_width // self.token_width)
        token_dim = self.token_height * self.token_width * C
        pos_embedding = nn.Parameter(torch.randn(1, num_patches, self.token_height * self.token_width * 2)).cuda()
        ########################################################################################
        to_patch_embedding = nn.Sequential(
            Rearrange('B C (h N_h) (w N_w) -> B (N_h N_w) (h w C)', h=self.token_height, w=self.token_width),
            nn.LayerNorm(token_dim),
            nn.Linear(token_dim, self.token_height * self.token_width * 2),
            nn.LayerNorm(self.token_height * self.token_width * 2)).cuda()
        ########################################################################################

        Q_2_in = to_patch_embedding(Q_2_in)
        K_2_in = to_patch_embedding(K_2_in)
        V_2_in = to_patch_embedding(V_2_in)
        b, n, _ = Q_2_in.shape
        Q = self.proj_drop(Q_2_in)
        Q += pos_embedding[:, :n]
        K = self.proj_drop(K_2_in)
        K += pos_embedding[:, :n]
        V = self.proj_drop(V_2_in)
        V += pos_embedding[:, :n]
        Q = self.to_q2(Q)
        K = self.to_k2(K)
        V = self.to_v2(V)
        q2, k2, v2 = map(lambda t: rearrange(t, 'b n (heads d) -> b heads n d', heads=self.heads), (Q, K, V))
        dots = torch.matmul(q2, k2.transpose(-1, -2)) * self.scale
        attn = self.attend(dots)
        attn = self.proj_drop(attn)
        out = torch.matmul(attn, v2)
        out = rearrange(out, 'b heads n d -> b n (heads d)')
        out = self.to_out(out)
        out = rearrange(out, 'B (N_h N_w) (h w C)->B C (h N_h) (w N_w)', h=self.token_height, w=self.token_width, N_h=patch_num_x, N_w=patch_num_y)
        return out
# -------------------------------------------------------
#           LR-HSI and RGB Fusion
# ------------------------------------------------------
class HRFusion(nn.Module):
    def __init__(self, *, token_size, dim, heads, pool='cls', dim_head=64, emb_dropout=0.):
        super(HRFusion, self).__init__()

        (self.token_height, self.token_width) = token_size
        self.token_size = token_size
        self.dim = dim
        assert pool in {'cls', 'mean'}, 'pool type must be either cls (cls token) or mean (mean pooling)'
        self.dropout = nn.Dropout(emb_dropout)
        self.to_latent = nn.Identity()
        self.MulCorssAttention = MulCorssAttention(dim, heads, dim_head, proj_drop=0.)
        self.Mspe = Mspe(dim, dim_head, heads)
        self.SpaFE = SpaFE()
        self.SpeFE = SpeFE(dim)
        self.conv_v = nn.Conv2d(in_channels=2 * dim, out_channels=dim, kernel_size=3, padding=1)
        self.BN=nn.BatchNorm2d(dim)

    def forward(self, x, mask, rgb):
        if mask is not None:
            LR_HSI = self.Mspe(x, mask)
        else:
            LR_HSI = x

        LR_HSI = LR_HSI.permute(0, 3, 1, 2)
        b, c, h, w = LR_HSI.shape
        V = LR_HSI
        K = torch.concat((LR_HSI, rgb), dim=1)
        K = self.BN(self.conv_v(K))
        Q = rgb
        atten = self.MulCorssAttention(V, K, Q)
        x = atten + K

        return x.permute(0, 2, 3, 1)


# -------------------------------------------------------
#           FeedForward
# ------------------------------------------------------
class FeedForward(nn.Module):
    def __init__(self, dim, mult=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(dim, dim * mult, 1, 1, bias=False),
            GELU(),
            nn.Conv2d(dim * mult, dim * mult, 3, 1, 1, bias=False, groups=dim * mult),
            GELU(),
            nn.Conv2d(dim * mult, dim, 1, 1, bias=False),
        )

    def forward(self, x):
        """
        x: [b,h,w,c]
        return out: [b,h,w,c]
        """
        out = self.net(x.permute(0, 3, 1, 2))
        return out.permute(0, 2, 3, 1)


# -------------------------------------------------------
#           Cascade  Transformer
# ------------------------------------------------------
class CascadeTransformer(nn.Module):
    def __init__(
            self, dim, dim_head, heads, num_blocks=1
    ):
        super().__init__()
        self.blocks = nn.ModuleList([])
        for _ in range(num_blocks):
            self.blocks.append(nn.ModuleList([
                Mspe(dim=dim, dim_head=dim_head, heads=heads),
                HRFusion(dim=dim, dim_head=dim_head, heads=heads, token_size=(16, 16)),
                PreNorm(dim, FeedForward(dim=dim))
            ]))

    def forward(self, x, mask, x_rgb):
        """
        x: [b,c,h,w]
        return out: [b,c,h,w]
        """
        x = x.permute(0, 2, 3, 1)
        for (attn1, attn2, ff) in self.blocks:
            x1 = attn1(x, mask) + x
            x2 = attn2(x1, mask=None, rgb=x_rgb) + x1
            x3 = ff(x2) + x2
        out = x3.permute(0, 3, 1, 2)
        return out


class HRFT(nn.Module):
    def __init__(self, dim=28, stage=3, num_blocks=None):
        super(HRFT, self).__init__()

        if num_blocks is None:
            num_blocks = [1, 1, 1]

        self.dim = dim
        self.stage = stage

        # Input projection
        self.embedding = nn.Conv2d(28, self.dim, 3, 1, 1, bias=False)

        # Encoder
        self.encoder_layers = nn.ModuleList([])
        dim_stage = dim
        for i in range(stage):
            self.encoder_layers.append(nn.ModuleList([
                CascadeTransformer(dim=dim_stage, num_blocks=num_blocks[i], dim_head=dim,
                                   heads=dim_stage // dim),
                nn.Conv2d(dim_stage, dim_stage * 2, 4, 2, 1, bias=False),
                nn.Conv2d(dim_stage, dim_stage * 2, 4, 2, 1, bias=False)
            ]))
            dim_stage *= 2
        # dim_stage

        # Bottleneck
        self.bottleneck = CascadeTransformer(dim=dim_stage, dim_head=dim, heads=dim_stage // dim,
                                             num_blocks=num_blocks[-1])

        # Decoder
        self.decoder_layers = nn.ModuleList([])
        for i in range(stage):
            self.decoder_layers.append(nn.ModuleList([
                nn.ConvTranspose2d(dim_stage, dim_stage // 2, stride=2, kernel_size=2, padding=0,
                                   output_padding=0),
                nn.Conv2d(dim_stage, dim_stage // 2, 1, 1, bias=False),
                CascadeTransformer(dim=dim_stage // 2, num_blocks=num_blocks[stage - 1 - i], dim_head=dim,
                                   heads=(dim_stage // 2) // dim)]))
            dim_stage //= 2

        # Output projection
        self.mapping = nn.Conv2d(self.dim, 28, 3, 1, 1, bias=False)

        #### activation function
        self.lrelu = nn.LeakyReLU(negative_slope=0.1, inplace=True)  #LeakyReLU

        self.SpaFE = SpaFE(n_fts=28)

    def forward(self, x, x_rgb, mask=None):
        """
        x: [b,c,h,w]
        return out:[b,c,h,w]
        """
        if mask == None:
            mask = torch.zeros((1, 28, 256, 310)).cuda()

        rgb_list = self.SpaFE(x_rgb)

        # Embedding
        fea = self.lrelu(self.embedding(x))
        # Encoder
        fea_encoder = []
        masks = []
        for i, (CascadeTransformer, FeaDownSample, MaskDownSample) in enumerate(self.encoder_layers):
            fea = CascadeTransformer(fea, mask, rgb_list[i])
            masks.append(mask)
            fea_encoder.append(fea)
            fea = FeaDownSample(fea)
            mask = MaskDownSample(mask)

        # Bottleneck
        fea = self.bottleneck(fea, mask, rgb_list[2])

        # Decoder
        for i, (FeaUpSample, Fution, LeWinBlcok) in enumerate(self.decoder_layers):
            fea = FeaUpSample(fea)
            fea = Fution(torch.cat([fea, fea_encoder[self.stage - 1 - i]], dim=1))
            mask = masks[self.stage - 1 - i]
            fea = LeWinBlcok(fea, mask, rgb_list[1 - i])

        # Mapping
        out = self.mapping(fea) + x

        return out


## Load pretrained model

In [51]:
print("Loading model from", MODEL_PTH)

# ── Fix for PyTorch >= 2.6 + multi-class pickle path ───────────────────────
# The .pth was saved when every class in HRFT.py lived under the training
# repo's module path: architecture.HRFT.<ClassName>.
# We register ALL classes from HRFT.py under that exact path so pickle can
# reconstruct them without any import errors.
import types

_hrft_classes = [
    HRFT, CascadeTransformer, HRFusion, Mspe, MaskGuidedMechanism,
    MulCorssAttention, SpaFE, SpeFE, FeedForward, PreNorm, GELU,
]

_arch_pkg  = types.ModuleType("architecture")
_arch_hrft = types.ModuleType("architecture.HRFT")
for _cls in _hrft_classes:
    setattr(_arch_hrft, _cls.__name__, _cls)
_arch_pkg.HRFT = _arch_hrft
sys.modules["architecture"]      = _arch_pkg
sys.modules["architecture.HRFT"] = _arch_hrft

model = torch.load(MODEL_PTH, map_location=DEVICE, weights_only=False)

# Unwrap DataParallel — trained on multi-GPU, run single-GPU on Kaggle
if isinstance(model, torch.nn.DataParallel):
    print("Unwrapping DataParallel wrapper ...")
    model = model.module

model = model.to(DEVICE)
model.eval()
print("Model ready on", next(model.parameters()).device)


Loading model from /kaggle/input/ifcasformer/pytorch/default/1/cave_test.pth
Unwrapping DataParallel wrapper ...
Model ready on cuda:0


## Inference — test() from test.py

In [52]:
# ════════════════════════════════════════════════════════════════════════════
# DIAGNOSTICS — run this before inference to verify data / mask / model
# ════════════════════════════════════════════════════════════════════════════
import torch

print("=" * 60)
print("1. DATA CHECK")
print(f"   test_data   shape : {test_data.shape}  min={test_data.min():.4f}  max={test_data.max():.4f}")
print(f"   label_rgb   shape : {label_rgb.shape}  min={label_rgb.min():.4f}  max={label_rgb.max():.4f}")

print()
print("2. MASK CHECK")
raw_mask = sio.loadmat(MASK_PATH)['CASSI']
print(f"   Raw mask_cave.mat : shape={raw_mask.shape}  dtype={raw_mask.dtype}")
print(f"   Values            : min={raw_mask.min():.4f}  max={raw_mask.max():.4f}  mean={raw_mask.mean():.4f}")
print(f"   Unique values     : {sorted(set(raw_mask.flatten().tolist()[:1000]))[:5]} ...")
print(f"   mask3d_batch      : {mask3d_batch.shape}")
print(f"   input_mask (Phi)  : {input_mask.shape}")

print()
print("3. MEASUREMENT CHECK")
test_gt    = test_data.cuda().float()
_label_rgb = label_rgb.cuda().float()
from torch import no_grad
input_meas = gen_meas_torch(test_gt, mask3d_batch, Y2H=True, mul_mask=False)
print(f"   input_meas        : {input_meas.shape}  min={input_meas.min():.4f}  max={input_meas.max():.4f}")

print()
print("4. MODEL CHECK")
print(f"   Model type        : {type(model).__name__}")
p = next(model.parameters())
print(f"   First param device: {p.device}  dtype={p.dtype}")
print(f"   First param stats : mean={p.data.mean():.6f}  std={p.data.std():.6f}")
n_params = sum(p.numel() for p in model.parameters())
print(f"   Total parameters  : {n_params:,}")

print()
print("5. QUICK FORWARD CHECK (1 scene)")
with no_grad():
    out1 = model(input_meas[:1], _label_rgb[:1], input_mask[:1] if input_mask.shape[0]>1 else input_mask)
print(f"   Output shape      : {out1.shape}")
print(f"   Output range      : min={out1.min():.4f}  max={out1.max():.4f}  mean={out1.mean():.4f}")
gt1 = test_gt[:1]
print(f"   GT    range       : min={gt1.min():.4f}  max={gt1.max():.4f}  mean={gt1.mean():.4f}")
diff = (out1 - gt1).abs()
print(f"   |out - gt|        : mean={diff.mean():.4f}  max={diff.max():.4f}")
print("=" * 60)


1. DATA CHECK
   test_data   shape : torch.Size([10, 28, 512, 512])  min=0.0000  max=1.0000
   label_rgb   shape : torch.Size([10, 3, 512, 512])  min=0.0000  max=1.0000

2. MASK CHECK
   Raw mask_cave.mat : shape=(512, 512)  dtype=uint8
   Values            : min=0.0000  max=1.0000  mean=0.5002
   Unique values     : [0, 1] ...
   mask3d_batch      : torch.Size([10, 28, 512, 512])
   input_mask (Phi)  : torch.Size([10, 28, 512, 566])

3. MEASUREMENT CHECK
   input_meas        : torch.Size([10, 28, 512, 512])  min=0.0000  max=1.5751

4. MODEL CHECK
   Model type        : HRFT
   First param device: cuda:0  dtype=torch.float32
   First param stats : mean=0.010505  std=0.071473
   Total parameters  : 42,554,604

5. QUICK FORWARD CHECK (1 scene)
   Output shape      : torch.Size([1, 28, 512, 512])
   Output range      : min=-0.1044  max=1.0126  mean=0.1451
   GT    range       : min=0.0000  max=1.0000  mean=0.1526
   |out - gt|        : mean=0.0118  max=0.2605


In [53]:
# Mirrors the test() function in test.py exactly
def test(model):
    test_gt    = test_data.cuda().float()
    _label_rgb = label_rgb.cuda().float()
    input_meas = gen_meas_torch(test_gt, mask3d_batch, Y2H=True, mul_mask=False)
    model.eval()

    with torch.no_grad():
        model_out = model(input_meas, _label_rgb, input_mask)

    pred  = np.transpose(model_out.detach().cpu().numpy(), (0, 2, 3, 1)).astype(np.float32)
    truth = np.transpose(test_gt.cpu().numpy(),            (0, 2, 3, 1)).astype(np.float32)

    results = []
    L, H, W, C = pred.shape
    print(f"{'Scene':<35} {'PSNR':>8} {'SSIM':>8} {'SAM':>8} {'ERGAS':>8}")
    print('-' * 75)
    for i in range(L):
        HSI = torch.tensor(pred[i, :, :, :]).permute(2, 0, 1)   # [C,H,W]
        gt  = torch.tensor(truth[i, :, :, :]).permute(2, 0, 1)
        psnr_v  = torch_psnr(HSI, gt)
        ssim_v  = torch_ssim(HSI, gt)
        sam_v   = SAM_GPU(HSI, gt)
        ergas_v = ERGAS(pred[i], truth[i])
        results.append({'scene': scene_names[i], 'psnr': float(psnr_v),
                        'ssim': float(ssim_v),   'sam':  float(sam_v),
                        'ergas': ergas_v})
        print(f'{scene_names[i]:<35} {float(psnr_v):>8.4f} {float(ssim_v):>8.4f}'
              f' {float(sam_v):>8.4f} {ergas_v:>8.4f}')
        mat_name = os.path.join(OUT_DIR, f'test_{i}.mat')
        print(f'Save reconstructed HSIs as {mat_name}.')
        sio.savemat(mat_name, {'truth': truth[i], 'pred': pred[i],
                               'psnr': float(psnr_v), 'ssim': float(ssim_v),
                               'sam': float(sam_v),   'ergas': ergas_v})
    model.train()
    return pred, truth, results

pred, truth, results = test(model)


Scene                                   PSNR     SSIM      SAM    ERGAS
---------------------------------------------------------------------------
scene01.mat                          34.6442   0.9548   5.2450   3.1192
Save reconstructed HSIs as /kaggle/working/results/test_0.mat.
scene04.mat                          34.2238   0.9646   4.7694   3.1518
Save reconstructed HSIs as /kaggle/working/results/test_1.mat.
scene05.mat                          35.8738   0.9461   6.8875   7.0436
Save reconstructed HSIs as /kaggle/working/results/test_2.mat.
scene06.mat                          35.3766   0.9618   6.1580   2.9753
Save reconstructed HSIs as /kaggle/working/results/test_3.mat.
scene07.mat                          42.3837   0.9865   5.9109   3.8046
Save reconstructed HSIs as /kaggle/working/results/test_4.mat.
scene08.mat                          39.1712   0.9719   4.7901   3.1003
Save reconstructed HSIs as /kaggle/working/results/test_5.mat.
scene09.mat                          30.46

In [54]:
# ── Summary (mirrors main() in test.py) ──────────────────────────────
mean_psnr  = np.mean([r['psnr']  for r in results])
mean_ssim  = np.mean([r['ssim']  for r in results])
mean_sam   = np.mean([r['sam']   for r in results])
mean_ergas = np.mean([r['ergas'] for r in results])

print('=' * 75)
print(f"{'MEAN':<35} {mean_psnr:>8.4f} {mean_ssim:>8.4f} {mean_sam:>8.4f} {mean_ergas:>8.4f}")

name = os.path.join(OUT_DIR, 'Test_result.mat')
print(f'Save reconstructed HSIs as {name}.')
sio.savemat(name, {'truth': truth, 'pred': pred,
                   'psnr': mean_psnr, 'ssim': mean_ssim,
                   'sam': mean_sam,   'ergas': mean_ergas})


MEAN                                 35.9799   0.9602   5.1497   3.5549
Save reconstructed HSIs as /kaggle/working/results/Test_result.mat.
